# Hypertower Repro Pipeline

This notebook documents the full run sequence used to reproduce current results.

## 0) Environment + Paths

- Activate `fundus_imaging` environment
- Run from repo root
- Confirm data paths:
  - `Papila/FundusImages`
  - `Papila/ClinicalData`
  - `Papila/ExpertsSegmentations`

In [ ]:
from pathlib import Path

required = [
    Path("Papila/FundusImages"),
    Path("Papila/ClinicalData"),
    Path("Papila/ExpertsSegmentations"),
    Path("REFUGE"),
]
for p in required:
    print(f"{p}:", "OK" if p.exists() else "MISSING")

## 1) Build UNet Manifest

In [ ]:
!python3 scripts/main/refuge/build_manifest.py --output manifest.csv

## 2) Train UNet Segmenter (per-image normalization)

Current tuned baseline:
- `--device cuda`
- `--batch-size 8`
- `--loader-workers 14`
- `--in-memory-cache`
- `--cache-workers 4`

In [ ]:
!python3 scripts/run_unet_segmenter.py \
  --manifest manifest.csv \
  --train --evaluate \
  --normalize per_image \
  --train-datasets refuge --val-datasets refuge --holdout-datasets refuge \
  --epochs 40 --batch-size 8 \
  --device cuda --loader-workers 14 \
  --in-memory-cache --cache-workers 4 \
  --checkpoint-dir models/v2/refuge/segmentation/per_image \
  --eval-output analysis_data/segmenter_eval/v2_refuge_per_image \
  --eval-metrics-path analysis_data/segmenter_eval/v2_refuge_per_image/metrics.csv

## 3) Run V2 Hypertower Modes (cropped with UNet)

Runs binary + multiclass across:
- `single`
- `ensemble`
- `bilateral`

In [ ]:
!python3 scripts/basic_analysis/compare_hypertower_modes.py \
  --eval-modes binary multiclass \
  --tower-modes single ensemble bilateral \
  --epochs 40 \
  --n-splits 5 \
  --batch-size 8 \
  --backbone refugelike \
  --img-crop-manifest manifest.csv \
  --img-crop-weights models/v2/refuge/segmentation/per_image/best.pt \
  --img-crop-normalize per_image \
  --img-crop-cache analysis_data/v2_crops_unet_refuge \
  --run-name v2_modes_full_40ep_5fold_unet_perimage

## 4) Quick Result Snapshot

In [ ]:
import json
from pathlib import Path

root = Path("analysis_data/v2_modes_full_40ep_5fold_unet_perimage")
summary = root / "summary.json"
if summary.exists():
    data = json.loads(summary.read_text())
    print("run_name:", data.get("run_name"))
    print("timestamp:", data.get("timestamp"))
    print("keys:", list(data.get("summaries", {}).keys()))
else:
    print("Summary not found:", summary)

## 5) Notes / Decisions

- Mixed-label patient handling used:
- Warmup settings used:
- Backbone / batch / workers used:
- Any deviations from default run: